In [8]:
# === Cellule 1 : imports et donnees ===
import numpy as np
import pandas as pd

df = pd.read_csv('../data/Artemis.csv')
print(df.shape)
print(df[['speed','hasAcceleration','hasPower','SOC_EB','SOC_PB']].describe())

(14530, 19)
              speed  hasAcceleration      hasPower        SOC_EB        SOC_PB
count  14530.000000     14530.000000  14530.000000  14530.000000  14530.000000
mean      10.660828         0.000082   3264.011708      0.587980      0.770584
std        8.094549         0.704734  10580.243967      0.247515      0.148476
min        0.000000        -4.080000 -46000.000000      0.199749      0.202598
25%        3.055556        -0.220000   -751.550000      0.364989      0.665434
50%       10.638889         0.000000   1053.660000      0.601149      0.778778
75%       17.076389         0.280000   8279.350000      0.808207      0.888395
max       30.972222         2.860000  44230.330000      1.000000      1.000000


In [9]:
# === Cellule 2 : constantes physiques (ontologie OntoHESS2.owl) ===
SOC_EB_MIN = 0.20
SOC_PB_MIN = 0.20  # pas dans l'ontologie, repris par defaut
P_EB_MIN = -6300.0
P_EB_MAX = 12600.0
P_PB_MIN = -52338.0
P_PB_MAX = 161040.0

In [10]:
# === Cellule 3 : fonctions d'appartenance trapezoidales ===
def trapmf(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    b = b if b > a else a + 1e-9
    c = c if c < d else d - 1e-9
    y = np.where((x >= b) & (x <= c), 1.0, 0.0)
    left = (x > a) & (x < b)
    y = np.where(left, (x - a) / (b - a), y)
    right = (x > c) & (x < d)
    y = np.where(right, (d - x) / (d - c), y)
    return y

# SOC : faible / moyen / eleve (memes bornes pour EB et PB)
def soc_faible(s): return trapmf(s, 0.20, 0.20, 0.30, 0.45)
def soc_moyen(s):  return trapmf(s, 0.30, 0.45, 0.60, 0.75)
def soc_eleve(s):  return trapmf(s, 0.60, 0.75, 1.00, 1.00)

# Pdem : forte_recharge / faible_recharge / nulle / faible_traction / forte_traction
def p_forte_recharge(p):  return trapmf(p, -46000, -46000, -22000, -8000)
def p_faible_recharge(p): return trapmf(p, -12000, -6000, -1000, 500)
def p_nulle(p):            return trapmf(p, -2000, -200, 200, 2000)
def p_faible_traction(p): return trapmf(p, -500, 1000, 6000, 12000)
def p_forte_traction(p):  return trapmf(p, 6000, 20000, 44230, 44230)

# acceleration : freinage / stable / moderee / forte
def a_freinage(a):  return trapmf(a, -4.08, -4.08, -1.2, -0.2)
def a_stable(a):    return trapmf(a, -0.4, -0.1, 0.1, 0.4)
def a_moderee(a):   return trapmf(a, 0.2, 0.6, 1.2, 1.8)
def a_forte(a):     return trapmf(a, 1.2, 1.8, 2.86, 2.86)

In [11]:
# === Cellule 4 : test des fonctions d'appartenance (verif visuelle rapide) ===
test_soc = np.array([0.20, 0.40, 0.60, 0.90])
print('soc_faible :', soc_faible(test_soc))
print('soc_moyen  :', soc_moyen(test_soc))
print('soc_eleve  :', soc_eleve(test_soc))

soc_faible : [0.         0.33333333 0.         0.        ]
soc_moyen  : [0.         0.66666667 1.         0.        ]
soc_eleve  : [0. 0. 0. 1.]


In [12]:
# === Cellule 5 : base de regles R1-R5, R7 (annexe B.1) ===
# R6 (limite convertisseur) et R8 (hors distribution) sont geres plus tard
# par le filtre physique et la fusion, pas par le controleur flou lui-meme.

def alpha_fuzzy_calc(soc_eb, soc_pb, pdem, accel):
    eb_faible = soc_faible(soc_eb)
    eb_moyen  = soc_moyen(soc_eb)
    eb_eleve  = soc_eleve(soc_eb)

    pb_faible = soc_faible(soc_pb)
    pb_moyen  = soc_moyen(soc_pb)
    pb_eleve  = soc_eleve(soc_pb)

    p_fr = p_forte_recharge(pdem)
    p_fbr = p_faible_recharge(pdem)
    p_n = p_nulle(pdem)
    p_fbt = p_faible_traction(pdem)
    p_ft = p_forte_traction(pdem)

    a_frein = a_freinage(accel)
    a_stab = a_stable(accel)

    traction = np.maximum(p_fbt, p_ft)
    recharge = np.maximum(np.maximum(p_fr, p_fbr), a_frein)
    pb_dispo = np.maximum(pb_moyen, pb_eleve)
    pb_rechargeable = np.maximum(pb_faible, pb_moyen)

    # R1 : SOC_PB faible et traction -> alpha faible (limiter decharge PB)
    w1, c1 = np.minimum(pb_faible, traction), 0.20
    # R2 : SOC_EB faible, PB dispo, traction -> alpha elevee
    w2, c2 = np.minimum(np.minimum(eb_faible, pb_dispo), traction), 0.75
    # R3 : forte traction et SOC_PB suffisant -> alpha elevee
    w3, c3 = np.minimum(p_ft, pb_dispo), 0.75
    # R4 : demande nulle et stable -> alpha faible (EB privilegie)
    w4, c4 = np.minimum(p_n, a_stab), 0.20
    # R5 : recharge et PB rechargeable -> alpha elevee (recup vers PB)
    w5, c5 = np.minimum(recharge, pb_rechargeable), 0.75
    # R7 : EB et PB faibles -> alpha moyenne (protection)
    w7, c7 = np.minimum(eb_faible, pb_faible), 0.50

    ws = np.stack([w1, w2, w3, w4, w5, w7])
    cs = np.array([c1, c2, c3, c4, c5, c7])
    wsum = ws.sum(axis=0)
    alpha = np.where(wsum > 1e-6, (ws * cs[:, None]).sum(axis=0) / np.maximum(wsum, 1e-9), 0.30)
    return alpha

In [13]:
# === Cellule 6 : scenarios de test S1-S7 (annexe B.2) ===
scenarios = {
    'S1 EB eleve, PB faible, demande moyenne': (0.85, 0.25, 5000, 0.1),
    'S2 EB eleve, PB eleve, forte accel':      (0.85, 0.85, 30000, 2.5),
    'S3 EB faible, PB eleve, forte demande':   (0.25, 0.85, 35000, 1.0),
    'S4 demande proche de zero':               (0.60, 0.60, 0, 0.0),
    'S5 freinage regeneratif':                 (0.60, 0.60, -10000, -2.0),
    'S6 demande superieure aux possibilites':  (0.60, 0.90, 44230, 2.0),
    'S7 etat jamais vu (ambigu)':              (0.50, 0.50, 20000, 0.5),
}
for name, (soc_eb, soc_pb, pdem, accel) in scenarios.items():
    a = alpha_fuzzy_calc(np.array([soc_eb]), np.array([soc_pb]), np.array([pdem]), np.array([accel]))[0]
    print(f'{name:45s} -> alpha_fuzzy = {a:.3f}')

S1 EB eleve, PB faible, demande moyenne       -> alpha_fuzzy = 0.200
S2 EB eleve, PB eleve, forte accel            -> alpha_fuzzy = 0.750
S3 EB faible, PB eleve, forte demande         -> alpha_fuzzy = 0.750
S4 demande proche de zero                     -> alpha_fuzzy = 0.338
S5 freinage regeneratif                       -> alpha_fuzzy = 0.750
S6 demande superieure aux possibilites        -> alpha_fuzzy = 0.300
S7 etat jamais vu (ambigu)                    -> alpha_fuzzy = 0.750


In [14]:
# === Cellule 7 : execution sur tout le cycle et sauvegarde ===
alpha = alpha_fuzzy_calc(df['SOC_EB'].values, df['SOC_PB'].values, df['hasPower'].values, df['hasAcceleration'].values)
df['alpha_fuzzy'] = alpha

print('alpha_fuzzy stats:')
print(df['alpha_fuzzy'].describe())
print('correlation with abs(Pdem):', np.corrcoef(df['alpha_fuzzy'], df['hasPower'].abs())[0,1])

df.to_csv('../data/artemis_alpha_fuzzy.csv', index=False)
print('saved to ../data/artemis_alpha_fuzzy.csv')

alpha_fuzzy stats:
count    14530.000000
mean         0.503036
std          0.237470
min          0.200000
25%          0.300000
50%          0.425708
75%          0.750000
max          0.750000
Name: alpha_fuzzy, dtype: float64
correlation with abs(Pdem): 0.48600930700214623
saved to ../data/artemis_alpha_fuzzy.csv
